Protecting an Investigation API with Entra ID

## Learning Objective

Understand how to protect an enterprise agent's API with Microsoft Entra ID, using delegated user access tokens and standard authentication middleware.

## Architecture

**User → Entra authentication → API access token → Token validation → Trusted principal → Claim-level authorization**

## What We'll Learn

- Why API registration and client registration are different.
- How delegated scopes limit a client's API permissions.
- Why access tokens must target the intended audience.
- How an authentication library validates Entra-issued tokens.
- Why application-specific authorization remains necessary.

## Experiment

Authenticate interactively, obtain a token for `Investigation.Read`, invoke the protected API, and compare authorized requests with rejected requests.

**Core principle: A valid identity and API scope do not imply access to every business resource.**

In [ ]:
import os
import msal
import requests
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
load_dotenv(ROOT / ".env")

tenant_id = os.environ["ENTRA_TENANT_ID"]
client_id = os.environ["ENTRA_CLIENT_ID"]
api_client_id = os.environ["ENTRA_API_CLIENT_ID"]

scope = f"api://{api_client_id}/Investigation.Read"

client = msal.PublicClientApplication(
    client_id=client_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)

result = client.acquire_token_interactive(
    scopes=[scope],
)

if "access_token" not in result:
    raise RuntimeError({
        "error": result.get("error"),
        "description": result.get("error_description"),
    })

access_token = result["access_token"]

print("Acquired delegated token for investigation API")

In [ ]:
import jwt

claims = jwt.decode(
    access_token,
    options={"verify_signature": False},
)

for field in ["aud", "iss", "tid", "oid", "scp", "azp", "ver"]:
    print(field, ":", claims.get(field))

In [ ]:
url = "http://localhost:8000/investigate"

headers = {
    "Authorization": f"Bearer {access_token}",
}

response = requests.post(
    url,
    json={"claim_id": "C-1042"},
    headers=headers,
    timeout=15,
)

print(response.status_code)
print(response.json())

assert response.status_code == 200
assert response.json()["claim"]["tenant_id"] == "northstar"

In [ ]:
response = requests.post(
    url,
    json={"claim_id": "C-1043"},
    headers=headers,
    timeout=15,
)

print(response.status_code, response.json())

assert response.status_code == 403

In [ ]:
response = requests.post(
    url,
    json={"claim_id": "C-1042"},
    timeout=15,
)

print(response.status_code)

assert response.status_code == 401